## 1. Model definition

$$q_\phi(z\mid x)=\mathcal N(\mu,\operatorname{diag}(e^\ell)),\qquad
z=\mu+e^{\ell/2}\odot\epsilon,\quad\epsilon\sim\mathcal N(0,I),\qquad
\hat x=f_\theta(z).$$

In [ ]:
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms

seed = 42
image_size = 128
latent_dim = 64
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
amp_enabled = device.type == "cuda"
torch.backends.cudnn.benchmark = amp_enabled
torch.set_float32_matmul_precision("high")


class VanillaVAE(nn.Module):
    def __init__(self, latent_dim=64, hidden_dims=(512, 256), image_size=128):
        super().__init__()
        pixels = image_size ** 2
        h1, h2 = hidden_dims
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(pixels, h1), nn.ReLU(),
            nn.Linear(h1, h2), nn.ReLU(),
        )
        self.fc_mu = nn.Linear(h2, latent_dim)
        self.fc_logvar = nn.Linear(h2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, h2), nn.ReLU(),
            nn.Linear(h2, h1), nn.ReLU(),
            nn.Linear(h1, pixels), nn.Sigmoid(),
            nn.Unflatten(1, (1, image_size, image_size)),
        )

    def encode(self, x):
        h = self.encoder(x)
        return self.fc_mu(h).float(), self.fc_logvar(h).float()

    def reparameterize(self, mu, logvar):
        return mu + torch.exp(0.5 * logvar) * torch.randn_like(mu)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        mu, logvar = self.encode(x)
        return self.decode(self.reparameterize(mu, logvar)), mu, logvar


model = VanillaVAE(latent_dim=latent_dim, hidden_dims=(512, 256), image_size=image_size).to(device)


## 2. Loss definition

$$\mathcal L_\beta=\frac1B\sum_{n=1}^{B}\left[
\|x_n-\hat x_n\|_2^2+
\frac\beta2\sum_{j=1}^{d}\left(\mu_{nj}^2+e^{\ell_{nj}}-1-\ell_{nj}\right)
\right].$$

$\beta=1$.

In [ ]:
def vae_loss(x_recon, x, mu, logvar, beta=1.0):
    recon = F.mse_loss(x_recon.float(), x.float(), reduction="sum") / x.size(0)
    mu, logvar = mu.float(), logvar.float()
    kl = 0.5 * (mu.square() + logvar.exp() - 1 - logvar).sum() / x.size(0)
    return recon + beta * kl, recon, kl

## 3. Dataset loading

In [ ]:
data_dir = Path("../VAE/archive/chest_xray/chest_xray")
batch_size = 32
num_workers = 4


def load_grayscale(path):
    with Image.open(path) as image:
        return image.convert("L")


transform = transforms.Compose([
    transforms.Resize((image_size, image_size), antialias=True),
    transforms.ToTensor(),
])
train_dataset = datasets.ImageFolder(data_dir / "train", transform=transform, loader=load_grayscale)
val_dataset = datasets.ImageFolder(data_dir / "val", transform=transform, loader=load_grayscale)
test_dataset = datasets.ImageFolder(data_dir / "test", transform=transform, loader=load_grayscale)


def make_loader(dataset, shuffle=False, num_workers=0):
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle,
        num_workers=num_workers, pin_memory=amp_enabled,
        persistent_workers=num_workers > 0,
        prefetch_factor=2 if num_workers > 0 else None,
        generator=torch.Generator().manual_seed(seed),
    )


train_loader = make_loader(train_dataset, shuffle=True, num_workers=num_workers)
val_loader = make_loader(val_dataset)
test_loader = make_loader(test_dataset, num_workers=num_workers)


## 4. Training

In [ ]:
@torch.inference_mode()
def evaluate_reconstruction(loader, noise_seed):
    model.eval()
    generator = torch.Generator(device=device).manual_seed(noise_seed)
    totals = torch.zeros(3, device=device)
    for images, _ in loader:
        images = images.to(device, non_blocking=True)
        mu, logvar = model.encode(images)
        noise = torch.randn(mu.shape, generator=generator, device=device)
        reconstructed = model.decode(mu + (0.5 * logvar).exp() * noise)
        _, recon, kl = vae_loss(reconstructed, images, mu, logvar)
        mean_recon = model.decode(mu)
        mean_sse = F.mse_loss(mean_recon, images, reduction="sum") / images.size(0)
        totals += torch.stack([recon, kl, mean_sse]) * images.size(0)
    recon, kl, mean_sse = (totals / len(loader.dataset)).cpu().tolist()
    return {"recon_sse": recon, "kl": kl, "loss": recon + kl,
            "mean_mse": mean_sse / image_size ** 2}


In [ ]:
epochs = 100
learning_rate = 0.001
warmup_epochs = 0
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
history = []

for epoch in range(1, epochs + 1):
    epoch_start = time.perf_counter()
    beta = min(1.0, epoch / warmup_epochs) if warmup_epochs else 1.0
    model.train()
    totals = torch.zeros(3, device=device)
    for images, _ in train_loader:
        images = images.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=amp_enabled):
            reconstructed, mu, logvar = model(images)
        loss, recon, kl = vae_loss(reconstructed, images, mu, logvar, beta)
        loss.backward()
        optimizer.step()
        totals += torch.stack([loss.detach(), recon.detach(), kl.detach()]) * images.size(0)
    train_loss, train_recon, train_kl = (totals / len(train_dataset)).cpu().tolist()
    validation = evaluate_reconstruction(val_loader, noise_seed=seed + 1)
    history.append({
        "epoch": epoch, "beta": beta,
        "train_loss": train_loss, "train_loss_beta1": train_recon + train_kl,
        "train_recon_sse": train_recon, "train_kl": train_kl,
        "val_loss_beta1": validation["loss"],
        "val_recon_sse": validation["recon_sse"], "val_kl": validation["kl"],
    })
    print(
        f"Epoch {epoch:03d}/{epochs} | beta={beta:.2f} | "
        f"Train={train_loss:.3f} | Recon={train_recon:.3f} | KL={train_kl:.3f} | "
        f"Val={validation['loss']:.3f} | {time.perf_counter() - epoch_start:.2f}s",
        flush=True,
    )


In [ ]:
epoch_numbers = [row["epoch"] for row in history]
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), constrained_layout=True)
for ax, key, title in zip(
    axes,
    ["loss_beta1", "recon_sse", "kl"],
    ["Reconstruction + KL", "Reconstruction SSE", "KL divergence"],
):
    ax.plot(epoch_numbers, [row[f"train_{key}"] for row in history], label="Train")
    ax.plot(epoch_numbers, [row[f"val_{key}"] for row in history], label="Validation")
    ax.set(title=title, xlabel="Epoch", ylabel="Loss per image")
    ax.grid(alpha=0.2)
    ax.legend()
plt.show()


In [ ]:
indices = []
for label in range(len(test_dataset.classes)):
    class_indices = np.flatnonzero(np.asarray(test_dataset.targets) == label)
    positions = np.linspace(0, len(class_indices) - 1, 4, dtype=int)
    indices.extend(class_indices[positions])

input_images = torch.stack([test_dataset[i][0] for i in indices]).to(device)
model.eval()
with torch.inference_mode():
    mu, _ = model.encode(input_images)
    reconstructed_images = model.decode(mu).cpu()
input_images = input_images.cpu()

fig, axes = plt.subplots(2, len(indices), figsize=(12, 3.4), constrained_layout=True)
for column, index in enumerate(indices):
    axes[0, column].imshow(input_images[column, 0], cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title(test_dataset.classes[test_dataset.targets[index]], fontsize=8)
    axes[1, column].imshow(reconstructed_images[column, 0], cmap="gray", vmin=0, vmax=1)
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
axes[0, 0].set_ylabel("Input")
axes[1, 0].set_ylabel("Reconstruction")
plt.show()


In [ ]:
test_metrics = evaluate_reconstruction(test_loader, noise_seed=seed + 2)
fixed_z = torch.randn(16, latent_dim, generator=torch.Generator().manual_seed(seed + 4)).to(device)
model.eval()
with torch.inference_mode():
    generated_images = model.decode(fixed_z).cpu()

fig, axes = plt.subplots(4, 4, figsize=(6, 6), constrained_layout=True)
for ax, image in zip(axes.flat, generated_images):
    ax.imshow(image[0], cmap="gray", vmin=0, vmax=1)
    ax.axis("off")
fig.suptitle("Samples from the standard-normal prior")
plt.show()


In [ ]:
from ignite.metrics import FID, InceptionScore

inception = models.inception_v3(weights=models.Inception_V3_Weights.DEFAULT).to(device).eval()
classifier = inception.fc
inception.fc = nn.Identity()
fid_metric = FID(num_features=2048, device=device)
is_metric = InceptionScore(num_features=1000, device=device)
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def inception_input(images):
    images = images.repeat(1, 3, 1, 1)
    images = F.interpolate(images, size=(299, 299), mode="bilinear", align_corners=False, antialias=True)
    return (images - mean) / std


generator = torch.Generator(device=device).manual_seed(seed + 3)
with torch.inference_mode():
    for real, _ in test_loader:
        real = real.to(device, non_blocking=True)
        z = torch.randn(real.size(0), latent_dim, device=device, generator=generator)
        fake = model.decode(z)
        real_features = inception(inception_input(real))
        fake_features = inception(inception_input(fake))
        fid_metric.update((fake_features, real_features))
        is_metric.update(classifier(fake_features).softmax(dim=1))

fid = float(fid_metric.compute())
inception_score = float(is_metric.compute())
print(f"Test MSE={test_metrics['mean_mse']:.6f} | FID={fid:.2f} | IS={inception_score:.3f}")
